In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import IntegerType, FloatType, DecimalType, BooleanType
from snowflake.snowpark.context import get_active_session
#Obteniendo la sesión activa
session = get_active_session()
#Definiendo el contexto
session.use_database("AI_IMPACT_DB")
session.use_schema("BRONZE")

In [ ]:
# Definiendo la ruta del archivo original (stage)
file_path = "@AI_IMPACT_DB.BRONZE.STG_RAW_DATA/ai_job_impact.csv"

# Definiendo el dataframe inicial
df_bronze = session.read.options({
    "field_delimiter": ",",
    "skip_header": 1
}).csv(file_path)

# Se genera la tabla bronze
df_bronze.write.mode("overwrite").save_as_table("AI_IMPACT_DB.BRONZE.RAW_AI_DATA_PY")

print(f"Capa Bronze completada con éxito. Se cargaron {df_bronze.count()} registros.")

df_bronze.show(10)

In [ ]:
# Cargando datos desde la capa bronze
df_silver = session.table("AI_IMPACT_DB.BRONZE.RAW_AI_DATA_PY")

# Aplicando transformaciones
df_silver_clean = df_silver.select(
    F.trim(F.col('"c1"')).alias("Employee_ID"),
    F.col('"c2"').cast(IntegerType()).alias("Age"),
    F.trim(F.col('"c3"')).alias("Gender"),
    F.trim(F.col('"c4"')).alias("Education_Level"),
    F.trim(F.col('"c5"')).alias("Industry"),
    F.trim(F.col('"c6"')).alias("Job_Role"),
    F.trim(F.col('"c9"')).alias("Automation_Risk"),
    #Transformando Yes/No a bool
    F.when(F.upper(F.trim(F.col('"c10"'))) == 'YES', True)
     .otherwise(False).cast(BooleanType()).alias("Upskilling_Required_Bool"),
    F.col('"c11"').cast(DecimalType(10,2)).alias("Salary_Before_AI"),
    F.col('"c12"').cast(DecimalType(10,2)).alias("Salary_After_AI"),
    F.col('"c16"').cast(IntegerType()).alias("Job_Satisfaction"),
    F.col('"c17"').cast(FloatType()).alias("Productivity_Change_Pct"),
    F.current_timestamp().alias("Load_Timestamp")
)

#Gardando la tabla transformada
df_silver_clean.write.mode("overwrite").save_as_table("AI_IMPACT_DB.SILVER.CLEANED_AI_DATA_PY")
print(f"Capa Silver completada con éxito. Se transformaron {df_silver_clean.count()} registros.")

df_silver_clean.show(10)

In [ ]:
#Cargando los datos desde la capa Silver
df_gold = session.table("AI_IMPACT_DB.SILVER.CLEANED_AI_DATA_PY")

# Insight #1 Impacto salarial por industria
salary_impact = df_gold.group_by("Industry").agg(
    F.round(F.avg("Salary_Before_AI"), 2).alias("Avg_Pre_AI"),
    F.round(F.avg("Salary_After_AI"), 2).alias("Avg_Post_AI")
).with_column("Net_Change", F.col("Avg_Post_AI") - F.col("Avg_Pre_AI"))

# Insight #2 Riesgo de automatización vs Nivel Educativo
education_risk = df_gold.group_by("Education_Level", "Automation_Risk").agg(
    F.count("Employee_ID").alias("Employee_Count")
).sort("Education_Level", F.col("Employee_Count").desc())

# Insight #3 Productividad y Felicidad
prod_vs_sati = df_gold.group_by("Job_Role").agg(
    F.round(F.avg("Productivity_Change_Pct"), 2).alias("Avg_Productivity_Gain"),
    F.round(F.avg("Job_Satisfaction"), 2).alias("Avg_Satisfaction")
).sort(F.col("Avg_Productivity_Gain").desc())

#Mostrando resultados
print("Insight 1: Impacto salarial por industria")
salary_impact.show()
print("Insight 2: Riesgo de Automatización por Nivel Educativo")
education_risk.show()
print("Insight 3: Productividad y Felicidad")
prod_vs_sati.show()